# 02 — Feature Engineering
**Football Betting Integrity Monitor**

Reads from `s_vladislavkivaev.mart_matches_clean` and builds the heavier, state/ordering-dependent
features in Python: per-league-season z-scores, cross-market divergence, and per-league rolling
temporal features — the work that needs ordering/state and doesn't belong in SQL.

**Design rule carried throughout this notebook**
Features are z-scored **per league-season**. The COVID season (19/20) is therefore already its own
baseline group — it's z-scored against itself like every other season, so no masking is needed here.
COVID rows are kept in the data; the `is_covid_season` flag is carried forward as a lever for
`03_modeling` (drop from Isolation Forest training, or use as a feature).

**Handoff to `03_modeling` (the H2-vs-H3 spine)**
This notebook deliberately preserves **both** representations of each signal:
- **natural-unit** values (scaled globally in modeling) → feed the **universal** Isolation Forest →
  mid-tier's wide-but-normal spreads sit far from the global mean → over-flagged. *This is H2.*
- **per-league-season z-scored** (`*_z`) values → feed the **tier-aware** leg → each league centred on
  its own normal → balanced false-positive rate. *This is H3.*

If we only kept the z-scored version, H2's failure mode would vanish — so Block 2 keeps both.

## Block 0 — Connect &amp; load

SQLAlchemy engine from `.env` (`sslmode=require`), read the mart, then sanity-check shape, coverage,
and that `is_covid_season` survived the dbt pass-through.

In [19]:
import os
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

pd.set_option("display.max_columns", 120)

In [20]:
load_dotenv()  # reads .env in the project root

DB_HOST     = os.environ["DB_HOST"]
DB_PORT     = os.environ.get("DB_PORT", "5432")
DB_NAME     = os.environ["DB_NAME"]
DB_USER     = os.environ["DB_USER"]
DB_PASSWORD = os.environ["DB_PASSWORD"]
DB_SCHEMA   = os.environ.get("DB_SCHEMA", "s_vladislavkivaev")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    connect_args={"sslmode": "require"},
)

In [21]:
df = pd.read_sql(f"SELECT * FROM {DB_SCHEMA}.mart_matches_clean", engine)
print(df.shape)
df.head()

(8915, 65)


,match_date,home_team,away_team,home_goals,away_goals,full_time_result,b365_open_h,b365_open_d,b365_open_a,ps_open_h,ps_open_d,ps_open_a,max_open_h,max_open_d,max_open_a,avg_open_h,avg_open_d,avg_open_a,b365_close_h,b365_close_d,b365_close_a,ps_close_h,ps_close_d,ps_close_a,max_close_h,max_close_d,max_close_a,avg_close_h,avg_close_d,avg_close_a,league,season,tier,is_covid_season,b365_drift_h,b365_drift_d,b365_drift_a,pinnacle_drift_h,pinnacle_drift_d,pinnacle_drift_a,opening_spread_h,opening_spread_d,opening_spread_a,closing_spread_h,closing_spread_d,closing_spread_a,max_opening_spread,max_closing_spread,spread_change_h,spread_change_d,spread_change_a,max_spread_change,b365_vs_market_h,b365_vs_market_d,b365_vs_market_a,implied_prob_sum_open,implied_prob_sum_close,season_quintile,b365_vs_ps_close_h,b365_vs_ps_close_d,b365_vs_ps_close_a,margin_tightened,overround_change,home_shortened,home_win
0,2019-08-16,Bayern Munich,Hertha,2,2,D,1.14,8.00,15.00,1.19,7.73,15.30,1.25,8.30,17.50,1.18,7.55,15.04,1.14,8.0,15.0,1.18,8.49,17.38,1.25,9.10,19.25,1.17,8.00,15.67,D1,1920,elite,True,0.00,0.00,0.00,-0.01,0.76,2.08,0.07,0.75,2.46,0.08,1.10,3.58,2.46,3.58,0.01,0.35,1.12,1.12,0.11,1.10,4.25,1.068860,1.068860,Q1\n(0-20%),-0.04,-0.49,-2.38,False,0.000000,False,False
1,2019-08-17,Dortmund,Augsburg,5,1,H,1.20,7.00,13.00,1.23,6.76,13.52,1.25,7.15,17.00,1.22,6.62,13.38,1.12,8.0,26.0,1.16,8.37,20.11,1.20,8.80,26.00,1.16,8.10,18.29,D1,1920,elite,True,-0.08,1.00,13.00,-0.07,1.61,6.59,0.03,0.53,3.62,0.04,0.70,7.71,3.62,7.71,0.01,0.17,4.09,4.09,0.08,0.80,0.00,1.053114,1.056319,Q1\n(0-20%),-0.04,-0.37,5.89,False,0.003205,True,True
2,2019-08-17,Freiburg,Mainz,3,0,H,2.25,3.25,3.40,2.23,3.45,3.44,2.26,3.49,3.65,2.20,3.37,3.36,2.55,3.3,2.7,2.74,3.30,2.77,2.74,3.38,2.96,2.61,3.29,2.78,D1,1920,elite,True,0.30,0.05,-0.70,0.51,-0.15,-0.67,0.06,0.12,0.29,0.13,0.09,0.18,0.29,0.18,0.07,-0.03,-0.11,0.07,0.19,0.08,0.26,1.046254,1.065558,Q1\n(0-20%),-0.19,0.00,-0.07,False,0.019303,False,True
3,2019-08-17,Leverkusen,Paderborn,3,2,H,1.25,6.00,12.00,1.29,6.07,10.57,1.31,6.40,12.25,1.28,5.97,10.27,1.22,6.5,11.0,1.27,6.69,10.45,1.29,7.05,13.00,1.25,6.52,10.70,D1,1920,elite,True,-0.03,0.50,-1.00,-0.02,0.62,-0.12,0.03,0.43,1.98,0.04,0.53,2.30,1.98,2.30,0.01,0.10,0.32,0.32,0.07,0.55,2.00,1.050000,1.064427,Q1\n(0-20%),-0.05,-0.19,0.55,False,0.014427,True,True
4,2019-08-17,Werder Bremen,Fortuna Dusseldorf,1,3,A,1.75,3.75,4.75,1.75,4.05,4.68,1.80,4.05,4.95,1.74,3.93,4.57,1.66,4.2,4.5,1.70,4.15,4.90,1.76,4.32,5.10,1.69,4.07,4.79,D1,1920,elite,True,-0.09,0.45,-0.25,-0.05,0.10,0.22,0.06,0.12,0.38,0.07,0.25,0.31,0.38,0.31,0.01,0.13,-0.07,0.13,0.10,0.12,0.60,1.048622,1.062727,Q1\n(0-20%),-0.04,0.05,-0.40,False,0.014106,True,False


In [22]:
# --- Sanity checks ---
df["match_date"] = pd.to_datetime(df["match_date"])
df = df.sort_values("match_date").reset_index(drop=True)

print("Rows:", len(df), "(source mart had 8,915)")
print("\nLeagues:", sorted(df["league"].unique()))
print("Seasons:", sorted(df["season"].unique()))

print("\nMatches per league x season:")
print(df.groupby(["league", "season"]).size().unstack(fill_value=0))

assert "is_covid_season" in df.columns, "is_covid_season missing — check the dbt pass-through"
print("\nCOVID rows:", int(df["is_covid_season"].sum()))

print("\nTier map:")
print(df[["tier", "league"]].drop_duplicates().sort_values(["tier", "league"]).to_string(index=False))

Rows: 8915 (source mart had 8,915)

Leagues: ['D1', 'E0', 'G1', 'T1']
Seasons: [np.int64(1920), np.int64(2021), np.int64(2122), np.int64(2223), np.int64(2324), np.int64(2425), np.int64(2526)]

Matches per league x season:
season  1920  2021  2122  2223  2324  2425  2526
league                                          
D1       306   306   306   306   306   306   306
E0       380   380   380   380   380   380   380
G1       240   239   240   238   240   233   236
T1       306   420   380   313   380   342   306

COVID rows: 1232

Tier map:
    tier league
   elite     D1
   elite     E0
mid_tier     G1
mid_tier     T1


In [23]:
# --- COVID baseline rule (defined ONCE, reused everywhere) ---
# Keep COVID rows in the data; exclude them when computing baseline mean/std.
baseline_mask = ~df["is_covid_season"].astype(bool)
print(f"Baseline rows (non-COVID): {int(baseline_mask.sum())} / {len(df)}")

Baseline rows (non-COVID): 7683 / 8915


In [24]:
# COVID flag. Under per-league-SEASON grouping, the 19/20 season is already its
# own baseline group, so it's isolated by the grouping itself — no masking needed
# in 02. Kept here as a lever for 03_modeling (drop from IF training, or use as a feature).
is_covid = df["is_covid_season"].astype(bool)
print(f"COVID rows: {int(is_covid.sum())} / {len(df)}")

COVID rows: 1232 / 8915


## Block 1 — The baseline engine (load-bearing)

One reusable function computes per-group mean/std and applies the z-score to every row in the group.
By default it uses all rows in each group (Option 1: each season, COVID included, is scored against
itself). It also accepts an optional `baseline_mask` to restrict which rows define the baseline —
unused in `02`, but available for `03_modeling`. Two guards:

- `std == 0` or `NaN` (thin season groups, esp. G1 / T1) → z set to `0.0`
- group with no baseline stats → z set to `0.0`

Default group key is `["league", "season"]`. A `["tier", "season"]` scheme is available for a
secondary view. Every downstream block calls this — so the normalization logic is debugged exactly once.

In [25]:
def zscore_by_group(df, cols, group_keys=("league", "season"),
                    baseline_mask=None, suffix="_z"):
    """
    Per-group z-score with a separate baseline population.

    Parameters
    ----------
    df : DataFrame
    cols : list[str]            columns to z-score
    group_keys : list[str]      grouping for the baseline (default league + season)
    baseline_mask : Series|None boolean aligned to df rows; True = row used to compute
                                mean/std. Pass ~is_covid_season to exclude COVID from baseline.
    suffix : str                appended to each new z-score column

    Returns
    -------
    DataFrame : copy of df with one `{col}{suffix}` column per input col.
                Guards: std == 0 / NaN / missing-group  ->  z = 0.0
    """
    group_keys = list(group_keys)
    cols = list(cols)
    out = df.reset_index(drop=True).copy()

    # 1) compute stats from the baseline population only
    base = out if baseline_mask is None else out.loc[baseline_mask.values]
    agg = base.groupby(group_keys)[cols].agg(["mean", "std"])
    agg.columns = [f"{c}__{stat}" for c, stat in agg.columns]
    agg = agg.reset_index()

    # 2) attach group stats to every row (left join preserves row order)
    merged = out.merge(agg, on=group_keys, how="left", sort=False)

    # 3) z-score with guards
    for col in cols:
        mean = merged[f"{col}__mean"]
        std  = merged[f"{col}__std"]
        z = (merged[col] - mean) / std
        z = z.where(std.notna() & (std != 0), 0.0).fillna(0.0)  # degenerate/undefined -> 0
        out[f"{col}{suffix}"] = z.to_numpy()

    return out

In [28]:
# --- Smoke test on one column (Option 1: no baseline_mask) ---
_test = zscore_by_group(df, cols=["b365_drift_h"])

print(_test[["league", "season", "b365_drift_h", "b365_drift_h_z"]].head())

print("\nAny inf in z? ", bool(np.isinf(_test["b365_drift_h_z"]).any()))
print("Any NaN in z? ", bool(_test["b365_drift_h_z"].isna().any()))

# Centring check: mean of z across every league-season group should be ~0
chk = _test.groupby(["league", "season"])["b365_drift_h_z"].mean()
print("\nMax |mean of z| across groups (expect ~0):", round(chk.abs().max(), 6))

  league  season  b365_drift_h  b365_drift_h_z
0     E0    1920          0.00       -0.006810
1     E0    1920          0.00       -0.006810
2     E0    1920          0.08        0.132133
3     E0    1920          0.40        0.687904
4     E0    1920          0.20        0.340547

Any inf in z?  False
Any NaN in z?  False

Max |mean of z| across groups (expect ~0): 0.0


In [27]:
chk = zscore_by_group(df, cols=["b365_drift_h"], baseline_mask=baseline_mask)
covid = df["is_covid_season"].astype(bool)
print("COVID rows, share z==0:     ", round((chk.loc[covid, "b365_drift_h_z"] == 0).mean(), 3))
print("Non-COVID rows, share z==0: ", round((chk.loc[~covid, "b365_drift_h_z"] == 0).mean(), 3))
print("\nNon-COVID z summary:\n", chk.loc[~covid, "b365_drift_h_z"].describe())

COVID rows, share z==0:      1.0
Non-COVID rows, share z==0:  0.003

Non-COVID z summary:
 count    7.683000e+03
mean    -4.624123e-18
std      9.969362e-01
min     -1.105378e+01
25%     -2.378979e-01
50%      0.000000e+00
75%      2.574367e-01
max      1.269332e+01
Name: b365_drift_h_z, dtype: float64


---
**Next — Block 2:** z-score the five signal families (drift, spread, implied-prob imbalance, CLV/cross-book,
reversal) through `zscore_by_group`, keeping the natural-unit columns alongside the new `*_z` columns.